# Thu thập giá OHLCV (`ingest_price_data`) — Luồng 1

**Đồ án:** Hệ thống Data Pipeline và tra cứu phân tích thị trường chứng khoán Việt Nam đa nguồn.

**Mục đích:** Tải lịch sử giá OHLCV có cấu trúc qua thư viện **vnstock** , chuẩn hóa/làm sạch, ghi vào **Data Lake** dạng Parquet theo partition Hive-style `date=YYYY-MM-DD` (ngày chạy pipeline).

**Nguồn `Quote`:** ưu tiên **KBS** (`PRIMARY_QUOTE_SOURCE`), fallback **VCI** (`FALLBACK_QUOTE_SOURCE`).

**Đầu ra:** `../data-lake/raw/price/date=<YYYY-MM-DD>/<Mã>.parquet` (một file mỗi mã); chỉ số: `../data-lake/raw/index/date=<YYYY-MM-DD>/<MÃ_CHỈ_SỐ>.parquet` (theo `INDEX_TICKERS`, cùng API `Quote.history` — xem `.cursor/rules/instructions.md`).

**Cách chạy:** Chọn kernel Python của project (venv), chạy lần lượt các ô: import → helper → định nghĩa hàm → `main()` (cổ phiếu) → ô **index** → các ô còn lại → QC cuối.


In [1]:
# vnstock Quote: PRIMARY = KBS, FALLBACK = VCI (nguồn ổn định).
# TCBS đã deprecated theo tài liệu vnstock — không dùng làm source.
import os

# Đặt UTF-8 để subprocess/thread đọc output không bị cp1252 trên Windows.
os.environ.setdefault("PYTHONUTF8", "1")
os.environ.setdefault("PYTHONIOENCODING", "utf-8")
os.environ.setdefault("LANG", "C.UTF-8")
os.environ.setdefault("LC_ALL", "C.UTF-8")
# Windows-specific encoding fix
os.environ["PYTHONLEGACYWINDOWSSTDIO"] = "0"

import logging
import time
from datetime import date
from pathlib import Path

import pandas as pd
from vnstock import Listing, Company, Finance, Quote

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

# ========== RATE LIMIT CONFIG ==========
RATE_LIMIT_RPM = 10  # 10 requests/phút
DELAY_BETWEEN_CATEGORIES_SEC = 60  # Pause 60s giữa các category
MAX_TICKERS_PER_RUN = 20  # Giới hạn 20 mã/lần chạy

TICKERS: list[str] = [
    # VN30 hiện tại (20 mã)
    "ACB","BCM","BID","BVH","CTG","FPT","GAS","GVR","HDB","HPG",
    "MBB","MSN","MWG","PLX","POW","SAB","SHB","SSI","STB","TCB",
    # # Thêm 30 mã đa ngành
    # "VCB","VHM","VIC","VJC","VNM","VPB","VRE","TPB","VIB","HVN",
    # "REE","PNJ","GMD","DGC","DPM","DCM","HSG","KDC","QNS","SCS",
    
    # "NVL","PDR","DXG","KDH","HDG","EVF","KBC","AGG","TCH","VPI",
]
PRIMARY_QUOTE_SOURCE = "kbs"
FALLBACK_QUOTE_SOURCE = "vci"

# Chỉ số thị trường VN — vnstock Quote.history (1D), mã symbol theo tài liệu vnstock (KBS/VCI).
INDEX_TICKERS: list[str] = [
    "VNINDEX",
    "VN30",
    "HNXINDEX",
    "HNX30",
    "UPCOMINDEX",
]

assert len(TICKERS) == 20, f"Expected 20 tickers, got {len(TICKERS)}"
assert len(set(TICKERS)) == len(TICKERS), "Duplicate tickers found"

In [2]:
import os
import sys
import warnings
import threading

os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

# Suppress encoding errors từ subprocess threads
original_hook = threading.excepthook
def custom_hook(args):
    if "UnicodeDecodeError" in str(args.exc_type):
        pass  # Ignore encoding errors from threads
    else:
        original_hook(args)
threading.excepthook = custom_hook

# Suppress non-critical warnings
warnings.filterwarnings("ignore")

print("[OK] UTF-8 guard enabled for current kernel session")

[OK] UTF-8 guard enabled for current kernel session


In [3]:
def save_raw(df: pd.DataFrame, category: str, run_date: str, name: str) -> str:
    """Lưu raw dạng song song Parquet + CSV (encoding utf-8-sig).

    Path:
      ../data-lake/raw/{category}/date={run_date}/{name}.parquet
    """
    base_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data-lake", "raw", category))
    partition = os.path.join(base_dir, f"date={run_date}")
    os.makedirs(partition, exist_ok=True)

    # Đối với ticker, luôn chuẩn hóa UPPER để đồng nhất tên file.
    if category in ("price", "index", "financial_ratio"):
        safe_name = str(name).upper().strip()
    else:
        safe_name = str(name).strip()

    parquet_path = os.path.join(partition, f"{safe_name}.parquet")
    csv_path = os.path.join(partition, f"{safe_name}.csv")

    try:
        df.to_parquet(parquet_path, engine="pyarrow", index=False)
    except ImportError:
        df.to_parquet(parquet_path, engine="fastparquet", index=False)


    logger.info("Đã ghi %s dòng → %s", len(df), parquet_path)
    return parquet_path


In [4]:
import re
import os
from pathlib import Path
from datetime import timedelta

# Rate limit state
_last_request_time = 0.0

def _ensure_ingestion_workdir():
    """Thiết lập working directory."""
    if not (Path.cwd() / "ingest_price_data.ipynb").is_file():
        os.chdir(Path.cwd() / "ingestion")

def _date_years_ago(ref_date, years):
    """Tính ngày cách đây N năm."""
    try:
        return ref_date.replace(year=ref_date.year - years)
    except ValueError:
        return ref_date.replace(year=ref_date.year - years, day=28)

def _wait_for_rate_limit():
    """Throttle requests."""
    global _last_request_time
    now = time.time()
    elapsed = now - _last_request_time
    min_interval = 60.0 / RATE_LIMIT_RPM
    if elapsed < min_interval:
        time.sleep(min_interval - elapsed)
    _last_request_time = time.time()

def _extract_wait_seconds(msg, default=60):
    """Extract wait time from error message."""
    match = re.search(r"(\d+)\s*(?:giây|s|second)", msg)
    return int(match.group(1)) if match else default

def transform_data(df):
    """Transform data: convert date, rename columns."""
    df = df.copy()
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    # vnstock/KBS thường trả cột thời gian là `time` khi không có tradingdate
    if "tradingdate" not in df.columns and "time" in df.columns:
        df = df.rename(columns={"time": "tradingdate"})
    df = df.rename(columns={
        "tradingdate": "tradingDate", "open": "open", "high": "high",
        "low": "low", "close": "close", "volume": "volume", "value": "value"
    })
    # Strip timestamp 07:00:00 thừa trong tradingDate
    if "tradingDate" in df.columns:
        df["tradingDate"] = pd.to_datetime(
            df["tradingDate"], errors="coerce"
        ).dt.strftime("%Y-%m-%d")
    # Tương tự nếu cột tên là "tradingdate" (lowercase)
    if "tradingdate" in df.columns:
        df["tradingdate"] = pd.to_datetime(
            df["tradingdate"], errors="coerce"
        ).dt.strftime("%Y-%m-%d")
    return df


def build_price_target_schema(df: pd.DataFrame, ticker: str, run_date: str) -> pd.DataFrame:
    """Giữ TẤT CẢ cột vnstock trả về, thêm metadata, đảm bảo schema tối thiểu."""
    out = df.copy()

    # Thêm metadata nếu chưa có
    if "ticker" not in out.columns:
        out.insert(0, "ticker", ticker)
    if "source" not in out.columns:
        out["source"] = PRIMARY_QUOTE_SOURCE
    if "ingested_at" not in out.columns:
        out["ingested_at"] = run_date

    # Đảm bảo các cột OHLCV tối thiểu tồn tại (điền NaN nếu thiếu)
    for col in ["open", "high", "low", "close", "volume", "value"]:
        if col not in out.columns:
            out[col] = pd.NA

    # Chuyển cột giá về numeric
    for col in ["open", "high", "low", "close", "volume", "value"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    # Thêm cờ is_suspicious nếu chưa có
    if "is_suspicious" not in out.columns:
        out["is_suspicious"] = (out["close"] < 0) | (out["high"] < out["low"])

    # Log các cột thực tế để theo dõi
    logger.info("%s — cột output: %s", ticker, out.columns.tolist())

    return out

def fetch_ticker_data(ticker, start, end):
    """Fetch price data từ vnstock."""
    sym = ticker.upper().strip()
    for src in (PRIMARY_QUOTE_SOURCE, FALLBACK_QUOTE_SOURCE):
        try:
            _wait_for_rate_limit()
            quote = Quote(source=src, symbol=sym)
            df = quote.history(start=start, end=end, interval="1D")
            if df is not None and not df.empty:
                logger.info("Lấy %s từ %s", sym, src.upper())
                return df.copy()
        except Exception as e:
            logger.warning("Fetch %s (%s) lỗi: %s", sym, src, str(e)[:100])
            continue
    return pd.DataFrame()


def fetch_index_data(index_code: str, start: str, end: str) -> pd.DataFrame:
    """Lịch sử chỉ số: cùng `Quote(...).history(..., interval='1D')` như cổ phiếu (vnstock free — instructions / 06-quote-price-api)."""
    return fetch_ticker_data(index_code, start, end)


def build_index_target_schema(df: pd.DataFrame, index_code: str, run_date: str) -> pd.DataFrame:
    """Schema giá chỉ số: giữ mọi cột API + metadata; đánh dấu instrument_type."""
    out = build_price_target_schema(df, index_code, run_date)
    if "instrument_type" not in out.columns:
        out["instrument_type"] = "index"
    return out


In [5]:
def main():
    """Thu thập giá OHLCV."""
    _ensure_ingestion_workdir()
    run_dt = date.today()
    run_date = run_dt.isoformat()
    start_dt = _date_years_ago(run_dt, 3)
    start_s = start_dt.isoformat()
    end_s = run_dt.isoformat()

    logger.info("=" * 60)
    logger.info("PRICE INGESTION: %s ~ %s", start_s, end_s)
    logger.info("=" * 60)

    total = len(TICKERS)
    for idx, ticker in enumerate(TICKERS):
        logger.info("[%d/%d] Fetching %s...", idx + 1, total, ticker)
        try:
            raw = fetch_ticker_data(ticker, start_s, end_s)
            if raw.empty:
                logger.warning("Bỏ qua %s (rỗng)", ticker)
                continue
            
            cleaned = transform_data(raw)
            final_df = build_price_target_schema(cleaned, ticker, run_date)
            save_raw(final_df, "price", run_date, ticker)
            logger.info("✓ %s: %d rows", ticker, len(final_df))
        except Exception as e:
            logger.error("Lỗi %s: %s", ticker, e)

    logger.info("✓ Hoàn tất\n")

main()


2026-04-04 15:58:42 [INFO] ============================================================
2026-04-04 15:58:42 [INFO] PRICE INGESTION: 2023-04-04 ~ 2026-04-04
2026-04-04 15:58:42 [INFO] ============================================================
2026-04-04 15:58:42 [INFO] [1/20] Fetching ACB...
2026-04-04 15:58:45 [INFO] Lấy ACB từ KBS
2026-04-04 15:58:45 [INFO] ACB — cột output: ['ticker', 'tradingDate', 'open', 'high', 'low', 'close', 'volume', 'source', 'ingested_at', 'value', 'is_suspicious']
2026-04-04 15:58:46 [INFO] Đã ghi 748 dòng → c:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\price\date=2026-04-04\ACB.parquet
2026-04-04 15:58:46 [INFO] ✓ ACB: 748 rows
2026-04-04 15:58:46 [INFO] [2/20] Fetching BCM...
2026-04-04 15:58:48 [INFO] Lấy BCM từ KBS
2026-04-04 15:58:48 [INFO] BCM — cột output: ['ticker', 'tradingDate', 'open', 'high', 'low', 'close', 'volume', 'source', 'ingested_at', 'value', 'is_suspicious']
2026-04-04 15:58:48 [INFO] Đã ghi 748 dòng → c:\Users\ACER\Do

In [6]:
# INDEX — chỉ số cơ sở (vnstock free: Quote + history 1D; xem .cursor/rules/instructions.md → Use Case 1 / 06-quote-price-api).
logger.info("⏸  Chờ %ss trước khi chuyển sang index...", DELAY_BETWEEN_CATEGORIES_SEC)
time.sleep(DELAY_BETWEEN_CATEGORIES_SEC)

_ensure_ingestion_workdir()
run_dt = date.today()
run_date = run_dt.isoformat()
start_dt = _date_years_ago(run_dt, 3)
start_s = start_dt.isoformat()
end_s = run_dt.isoformat()

logger.info("=" * 60)
logger.info("INDEX INGESTION: %s ~ %s", start_s, end_s)
logger.info("=" * 60)

n_idx = len(INDEX_TICKERS)
for j, index_code in enumerate(INDEX_TICKERS):
    logger.info("[%d/%d] Fetching index %s...", j + 1, n_idx, index_code)
    try:
        raw_i = fetch_index_data(index_code, start_s, end_s)
        if raw_i.empty:
            logger.warning("Bỏ qua index %s (rỗng)", index_code)
            continue
        cleaned_i = transform_data(raw_i)
        final_i = build_index_target_schema(cleaned_i, index_code, run_date)
        save_raw(final_i, "index", run_date, index_code)
        logger.info("✓ %s: %d rows", index_code, len(final_i))
    except Exception as e:
        logger.error("Lỗi index %s: %s", index_code, e)

logger.info("✓ Hoàn tất index\n")

2026-04-04 16:00:36 [INFO] ⏸  Chờ 60s trước khi chuyển sang index...
2026-04-04 16:01:36 [INFO] ============================================================
2026-04-04 16:01:36 [INFO] INDEX INGESTION: 2023-04-04 ~ 2026-04-04
2026-04-04 16:01:36 [INFO] ============================================================
2026-04-04 16:01:36 [INFO] [1/5] Fetching index VNINDEX...
2026-04-04 16:01:37 [INFO] Lấy VNINDEX từ KBS
2026-04-04 16:01:37 [INFO] VNINDEX — cột output: ['ticker', 'tradingDate', 'open', 'high', 'low', 'close', 'volume', 'source', 'ingested_at', 'value', 'is_suspicious']
2026-04-04 16:01:37 [INFO] Đã ghi 748 dòng → c:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\index\date=2026-04-04\VNINDEX.parquet
2026-04-04 16:01:37 [INFO] ✓ VNINDEX: 748 rows
2026-04-04 16:01:37 [INFO] [2/5] Fetching index VN30...
2026-04-04 16:01:43 [INFO] Lấy VN30 từ KBS
2026-04-04 16:01:43 [INFO] VN30 — cột output: ['ticker', 'tradingDate', 'open', 'high', 'low', 'close', 'volume', 'source', 

In [7]:
# FOREIGN TRADING: KBS free tier không hỗ trợ dữ liệu khối ngoại
# qua quote.history(). Category này bị bỏ qua.
# Nếu cần: xem xét vnstock_data (sponsored tier) hoặc
# lấy trực tiếp từ trang SSC/HOSE.
logger.info("foreign_trading: bỏ qua — KBS không hỗ trợ qua free tier")

2026-04-04 16:02:01 [INFO] foreign_trading: bỏ qua — KBS không hỗ trợ qua free tier


In [8]:
def fetch_listing() -> pd.DataFrame:
    """Lấy danh sách mã niêm yết (master data)."""
    logger.info("⏸  Chờ %ss trước khi chuyển sang company listing...", DELAY_BETWEEN_CATEGORIES_SEC)
    time.sleep(DELAY_BETWEEN_CATEGORIES_SEC)

    def _standardize_columns(df_in: pd.DataFrame) -> pd.DataFrame:
        out = df_in.copy()
        out.columns = [str(c).strip().lower() for c in out.columns]
        return out

    sources: tuple[str, ...] = (PRIMARY_QUOTE_SOURCE, FALLBACK_QUOTE_SOURCE)
    last_err: Exception | None = None
    df_base = pd.DataFrame()

    for src in sources:
        try:
            listing = Listing(source=src)
            df = listing.all_symbols()
            if df is None or (hasattr(df, "empty") and df.empty):
                logger.warning("all_symbols() trả rỗng với Listing source=%s", src)
                continue

            df_base = _standardize_columns(df)
            logger.info(
                "Lấy Listing thành công (%s mã) từ source=%s",
                len(df_base),
                src.upper(),
            )
            break

        except Exception as e:
            last_err = e
            logger.warning("Lỗi fetch_listing source=%s: %s", src, e)
            continue

    if df_base.empty:
        logger.warning("Không lấy được Listing: %s", last_err)
        return pd.DataFrame()

    exchanges = ["HOSE", "HNX", "UPCOM"]
    exchange_dfs = []

    for exc in exchanges:
        try:
            listing_obj = Listing(source=PRIMARY_QUOTE_SOURCE)
            if hasattr(listing_obj, "symbols_by_exchange"):
                df_exc = listing_obj.symbols_by_exchange(exchange=exc)
            elif hasattr(listing_obj, "symbols_by_group"):
                df_exc = listing_obj.symbols_by_group(group=exc)
            else:
                continue

            if df_exc is not None and not df_exc.empty:
                df_exc = df_exc.copy()
                df_exc["exchange"] = exc
                exchange_dfs.append(df_exc)
                logger.info("Lấy %s symbols từ %s", len(df_exc), exc)
        except Exception as e:
            logger.warning("Không lấy được %s: %s", exc, e)

    if exchange_dfs:
        df_all_exchanges = pd.concat(exchange_dfs, ignore_index=True)
        df_all_exchanges.columns = [
            c.strip().lower().replace(" ", "_") for c in df_all_exchanges.columns
        ]

        df_base_merge = df_base.copy()
        df_base_merge.columns = [c.strip().lower() for c in df_base_merge.columns]
        ticker_col = "symbol" if "symbol" in df_base_merge.columns else df_base_merge.columns[0]

        exc_ticker_col = None
        for cname in ["symbol", "ticker", "code"]:
            if cname in df_all_exchanges.columns:
                exc_ticker_col = cname
                break

        if exc_ticker_col and "organ_name" in df_base_merge.columns:
            df_merged = pd.merge(
                df_all_exchanges,
                df_base_merge[[ticker_col, "organ_name"]].rename(
                    columns={ticker_col: exc_ticker_col}
                ),
                on=exc_ticker_col,
                how="left",
            )
            df_result = df_merged
        elif exc_ticker_col:
            df_result = df_all_exchanges
        else:
            df_result = df_all_exchanges
    else:
        logger.warning(
            "Không lấy được thông tin exchange riêng — "
            "dùng all_symbols() với exchange=UNKNOWN"
        )
        df_result = df_base.copy()
        df_result["exchange"] = "UNKNOWN"

    # Fix 1 — Lọc chỉ giữ cổ phiếu thường (type=stock hoặc id=1)
    before_filter = len(df_result)

    if "type" in df_result.columns:
        df_stock = df_result[df_result["type"] == "stock"].copy()
    elif "id" in df_result.columns:
        df_stock = df_result[df_result["id"] == 1].copy()
    else:
        df_stock = df_result.copy()

    after_filter = len(df_stock)
    logger.info(
        "Lọc stock: %s → %s dòng (bỏ %s bond/fund/future)",
        before_filter,
        after_filter,
        before_filter - after_filter,
    )
    df_result = df_stock

    # Fix 2 — Exchange không tin cậy: cùng dataset qua symbols_by_exchange,
    # hoặc chỉ một giá trị duy nhất (vd. toàn HOSE)
    if "exchange" in df_result.columns:
        unique_exchanges = df_result["exchange"].dropna().unique().tolist()
        if exchange_dfs:
            logger.warning(
                "exchange (HOSE/HNX/UPCOM) từ symbols_by_exchange dùng chung "
                "dataset — không tin cậy, đặt UNKNOWN cho %s mã",
                len(df_result),
            )
            df_result["exchange"] = "UNKNOWN"
        elif len(unique_exchanges) == 1 and str(unique_exchanges[0]).upper() != "UNKNOWN":
            logger.warning(
                "exchange chỉ có 1 giá trị '%s' cho toàn bộ %s mã "
                "— không tin cậy, đặt lại thành UNKNOWN",
                unique_exchanges[0],
                len(df_result),
            )
            df_result["exchange"] = "UNKNOWN"

    # Fix 3 — Dọn cột thừa và thứ tự cột
    if "organ_name_x" in df_result.columns:
        df_result = df_result.rename(columns={"organ_name_x": "organ_name"})
    if "organ_name_y" in df_result.columns:
        df_result = df_result.drop(columns=["organ_name_y"])
    if "id" in df_result.columns:
        df_result = df_result.drop(columns=["id"])

    # Fix 4 — crawled_at chuẩn YYYY-MM-DD (trước khi sắp xếp cột)
    df_result["crawled_at"] = date.today().strftime("%Y-%m-%d")

    priority_cols = [
        "symbol",
        "organ_name",
        "en_organ_name",
        "exchange",
        "type",
        "crawled_at",
    ]
    existing = [c for c in priority_cols if c in df_result.columns]
    other = [c for c in df_result.columns if c not in priority_cols]
    df_result = df_result[existing + other]

    logger.info("Listing columns cuối: %s", df_result.columns.tolist())
    if "exchange" in df_result.columns:
        logger.info(
            "Phân bổ theo sàn:\n%s",
            df_result["exchange"].value_counts().to_string(),
        )

    return df_result


# --- Run Listing ---
logger.info("=" * 70)
logger.info("LISTING (Master data)")
logger.info("=" * 70)

_ensure_ingestion_workdir()
df_listing = fetch_listing()

if df_listing.empty:
    logger.warning("Bỏ qua lưu Listing vì DataFrame rỗng.")
else:
    # Thêm debug print để kiểm tra cột
    print("=== LISTING COLUMNS ===")
    print(df_listing.columns.tolist())
    print(df_listing.head(3).to_string())
    
    # Giữ TẤT CẢ cột vnstock trả về, không drop gì cả
    # Chỉ thêm cột crawled_at
    df_listing["crawled_at"] = date.today().isoformat()
    
    # Chuẩn hóa tên cột về lowercase + strip
    df_listing.columns = [c.strip().lower().replace(" ", "_") 
                           for c in df_listing.columns]
    
    # Log để biết có những cột gì
    logger.info("Listing columns: %s", df_listing.columns.tolist())
    logger.info("Listing shape: %s", df_listing.shape)
    
    # Lưu vào partition master thay vì date-based
    try:
        base_dir = os.path.abspath(
            os.path.join(os.getcwd(), "..", "data-lake", "raw", "listing", "master")
        )
        os.makedirs(base_dir, exist_ok=True)
        
        parquet_path = os.path.join(base_dir, "listing.parquet")
        csv_path = os.path.join(base_dir, "listing.csv")
        
        try:
            df_listing.to_parquet(parquet_path, engine="pyarrow", index=False)
        except ImportError:
            df_listing.to_parquet(parquet_path, engine="fastparquet", index=False)
        df_listing.to_csv(csv_path, index=False, encoding="utf-8-sig")
        
        logger.info("✓ Đã ghi %s dòng → %s", len(df_listing), parquet_path)
        logger.info("✓ Đã xuất %s dòng → %s", len(df_listing), csv_path)
    except Exception:
        logger.exception("✗ Lỗi khi lưu Listing")

2026-04-04 16:02:01 [INFO] ======================================================================
2026-04-04 16:02:01 [INFO] LISTING (Master data)
2026-04-04 16:02:01 [INFO] ======================================================================
2026-04-04 16:02:01 [INFO] ⏸  Chờ 60s trước khi chuyển sang company listing...


2026-04-04 16:03:01 [INFO] Lấy Listing thành công (1544 mã) từ source=KBS
2026-04-04 16:03:02 [INFO] Lấy 1942 symbols từ HOSE
2026-04-04 16:03:02 [INFO] Lấy 1942 symbols từ HNX
2026-04-04 16:03:03 [INFO] Lấy 1942 symbols từ UPCOM
2026-04-04 16:03:03 [INFO] Lọc stock: 5826 → 4632 dòng (bỏ 1194 bond/fund/future)
2026-04-04 16:03:03 [WARNING] exchange (HOSE/HNX/UPCOM) từ symbols_by_exchange dùng chung dataset — không tin cậy, đặt UNKNOWN cho 4632 mã
2026-04-04 16:03:03 [INFO] Listing columns cuối: ['symbol', 'organ_name', 'en_organ_name', 'exchange', 'type', 'crawled_at']
2026-04-04 16:03:03 [INFO] Phân bổ theo sàn:
exchange
UNKNOWN    4632
2026-04-04 16:03:03 [INFO] Listing columns: ['symbol', 'organ_name', 'en_organ_name', 'exchange', 'type', 'crawled_at']
2026-04-04 16:03:03 [INFO] Listing shape: (4632, 6)
2026-04-04 16:03:03 [INFO] ✓ Đã ghi 4632 dòng → c:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\listing\master\listing.parquet
2026-04-04 16:03:03 [INFO] ✓ Đã xuất 4632 

=== LISTING COLUMNS ===
['symbol', 'organ_name', 'en_organ_name', 'exchange', 'type', 'crawled_at']
  symbol                 organ_name                                en_organ_name exchange   type  crawled_at
0    DPP         CTCP Dược Đồng Nai  Dong Nai Pharmaceutical Joint Stock Company  UNKNOWN  stock  2026-04-04
1    SDA         CTCP Simco Sông Đà            SIMCO Song Da Joint Stock Company  UNKNOWN  stock  2026-04-04
2    CLH  CTCP Xi măng La Hiên VVMI      VVMI La Hien Cement Joint Stock Company  UNKNOWN  stock  2026-04-04


In [9]:
def fetch_company_overview(ticker: str) -> dict:
    """Lấy thông tin tổng quan doanh nghiệp cho một mã.

    Không raise exception; lỗi sẽ trả về dict {'ticker': ticker}.
    """
    sym = ticker.upper().strip()
    try:
        company = Company(source=PRIMARY_QUOTE_SOURCE, symbol=sym)

        result = None
        try:
            if hasattr(company, "overview"):
                result = company.overview()
            elif hasattr(company, "profile"):
                result = company.profile()
            else:
                logger.warning("Company: không có method overview/profile (ticker=%s)", sym)
                result = {}
        except Exception as e:
            logger.warning("Company method lỗi %s: %s", sym, e)
            result = {}

        # Chuẩn hóa về dict
        if result is None:
            out: dict = {}
        elif isinstance(result, dict):
            out = dict(result)
        elif isinstance(result, pd.Series):
            out = result.to_dict()
        elif isinstance(result, pd.DataFrame):
            out = result.iloc[0].to_dict() if not result.empty else {}
        else:
            try:
                out = dict(result)
            except Exception:
                out = {}

        out["ticker"] = sym
        return out

    except Exception as e:
        logger.warning("fetch_company_overview lỗi %s: %s", sym, e)
        return {"ticker": sym}


# --- Run company overview ---
logger.info("⏸  Chờ %ss trước khi chuyển sang company overview...", DELAY_BETWEEN_CATEGORIES_SEC)
time.sleep(DELAY_BETWEEN_CATEGORIES_SEC)

_ensure_ingestion_workdir()
run_dt = date.today()
run_date = run_dt.isoformat()

logger.info("=" * 70)
logger.info("COMPANY OVERVIEW | %s mã", len(TICKERS))
logger.info("=" * 70)

rows: list[dict] = []
total = len(TICKERS)
for idx, ticker in enumerate(TICKERS):
    logger.info("[%s/%s] Đang tải company %s...", idx + 1, total, ticker)
    try:
        rows.append(fetch_company_overview(ticker))
        logger.info("✓ OK company %s", ticker)
    except Exception:
        logger.exception("✗ Lỗi khi xử lý company %s — tiếp tục", ticker)

    if idx < total - 1:
        time.sleep(0.5)

company_df = pd.DataFrame(rows)

try:
    if company_df.empty:
        logger.warning("company_df rỗng — bỏ qua lưu.")
    else:
        # Xử lý các cột text dài: replace newline bằng space
        text_cols = company_df.select_dtypes(include="object").columns
        for col in text_cols:
            company_df[col] = (
                company_df[col]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.replace("\r", " ", regex=False)
                .str.strip()
            )
        
        save_raw(company_df, "company", run_date, "company_overview")
        logger.info("✓ Đã lưu company overview\n")
except Exception:
    logger.exception("✗ Lỗi khi lưu company_overview")

2026-04-04 16:03:03 [INFO] ⏸  Chờ 60s trước khi chuyển sang company overview...
2026-04-04 16:04:03 [INFO] ======================================================================
2026-04-04 16:04:03 [INFO] COMPANY OVERVIEW | 20 mã
2026-04-04 16:04:03 [INFO] ======================================================================
2026-04-04 16:04:03 [INFO] [1/20] Đang tải company ACB...
2026-04-04 16:04:03 [INFO] ✓ OK company ACB
2026-04-04 16:04:04 [INFO] [2/20] Đang tải company BCM...
2026-04-04 16:04:04 [INFO] ✓ OK company BCM
2026-04-04 16:04:04 [INFO] [3/20] Đang tải company BID...
2026-04-04 16:04:05 [INFO] ✓ OK company BID
2026-04-04 16:04:05 [INFO] [4/20] Đang tải company BVH...
2026-04-04 16:04:06 [INFO] ✓ OK company BVH
2026-04-04 16:04:06 [INFO] [5/20] Đang tải company CTG...
2026-04-04 16:04:06 [INFO] ✓ OK company CTG
2026-04-04 16:04:07 [INFO] [6/20] Đang tải company FPT...
2026-04-04 16:04:07 [INFO] ✓ OK company FPT
2026-04-04 16:04:08 [INFO] [7/20] Đang tải company GAS...
20

In [10]:
def fetch_financial_ratio(ticker: str) -> pd.DataFrame:
    """Lấy financial ratio theo quý (fallback year nếu lỗi)."""
    sym = ticker.upper().strip()
    try:
        finance = Finance(source=PRIMARY_QUOTE_SOURCE, symbol=sym)

        df = pd.DataFrame()
        try:
            df = finance.ratio(period="quarter")
        except Exception:
            df = finance.ratio(period="year")

        if df is None or (hasattr(df, "empty") and df.empty):
            return pd.DataFrame()

        if not isinstance(df, pd.DataFrame):
            df = pd.DataFrame(df)

        out = df.copy()
        out["ticker"] = sym
        return out

    except Exception as e:
        logger.warning("fetch_financial_ratio lỗi %s: %s", sym, e)
        return pd.DataFrame()


# --- Run financial_ratio ---
logger.info("⏸  Chờ %ss trước khi chuyển sang financial_ratio...", DELAY_BETWEEN_CATEGORIES_SEC)
time.sleep(DELAY_BETWEEN_CATEGORIES_SEC)

_ensure_ingestion_workdir()
run_dt = date.today()
run_date = run_dt.isoformat()

logger.info("=" * 70)
logger.info("FINANCIAL RATIO | %s mã (theo quarter)", len(TICKERS))
logger.info("=" * 70)

total = len(TICKERS)
for idx, ticker in enumerate(TICKERS):
    logger.info("[%s/%s] Đang tải financial ratio %s...", idx + 1, total, ticker)
    try:
        df_ratio = fetch_financial_ratio(ticker)
        if df_ratio.empty:
            logger.warning("Bỏ qua %s — financial_ratio rỗng", ticker)
        else:
            save_raw(df_ratio, "financial_ratio", run_date, ticker)
            logger.info("✓ Thành công ratio %s (%s dòng)", ticker, len(df_ratio))
    except Exception:
        logger.exception("✗ Lỗi khi xử lý financial_ratio %s — tiếp tục", ticker)

    if idx < total - 1:
        time.sleep(0.5)  # Giảm từ 1s xuống 0.5s

logger.info("✓ Hoàn tất financial_ratio cho %s mã.\n", total)

2026-04-04 16:04:19 [INFO] ⏸  Chờ 60s trước khi chuyển sang financial_ratio...
2026-04-04 16:05:19 [INFO] ======================================================================
2026-04-04 16:05:19 [INFO] FINANCIAL RATIO | 20 mã (theo quarter)
2026-04-04 16:05:19 [INFO] ======================================================================
2026-04-04 16:05:19 [INFO] [1/20] Đang tải financial ratio ACB...
2026-04-04 16:05:19 [INFO] Loaded 162 built-in KBS field mappings
2026-04-04 16:05:19 [INFO] Loaded 162 built-in KBS field mappings
2026-04-04 16:05:20 [INFO] Đã ghi 32 dòng → c:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\financial_ratio\date=2026-04-04\ACB.parquet
2026-04-04 16:05:20 [INFO] ✓ Thành công ratio ACB (32 dòng)
2026-04-04 16:05:20 [INFO] [2/20] Đang tải financial ratio BCM...
2026-04-04 16:05:20 [INFO] Loaded 162 built-in KBS field mappings
2026-04-04 16:05:20 [INFO] Loaded 162 built-in KBS field mappings
2026-04-04 16:05:21 [INFO] Đã ghi 32 dòng → c:\Users\A

In [11]:
def fetch_price_board(tickers_list: list[str]) -> pd.DataFrame:
    """Thu thập snapshot bảng giá tham chiếu/trần/sàn.

    Nếu API/method không khả dụng sẽ log warning và trả DataFrame rỗng.
    """
    from datetime import datetime, time as dtime
    import re

    required_norm_map = {
        "ticker": "ticker",
        "ref_price": "ref_price",
        "ceiling_price": "ceiling_price",
        "floor_price": "floor_price",
        "match_price": "match_price",
        "match_volume": "match_volume",
        "foreign_buy_vol": "foreign_buy_vol",
        "foreign_sell_vol": "foreign_sell_vol",
        "bid_price1": "bid_price1",
        "bid_vol1": "bid_vol1",
        "ask_price1": "ask_price1",
        "ask_vol1": "ask_vol1",
    }

    def _norm_col(s: str) -> str:
        return re.sub(r"[\s_]+", "", str(s).strip().lower())

    # Chỉ chạy trong giờ giao dịch theo yêu cầu.
    now_t = datetime.now().time()
    if not (dtime(9, 0) <= now_t <= dtime(15, 0)):
        logger.info("Ngoài giờ giao dịch (now=%s) — bỏ qua price_board.", now_t.strftime("%H:%M:%S"))
        return pd.DataFrame()

    if not tickers_list:
        return pd.DataFrame()

    try:
        probe = Quote(source=PRIMARY_QUOTE_SOURCE, symbol=tickers_list[0])
        if not hasattr(probe, "price_board"):
            logger.warning("Method `Quote.price_board()` không khả dụng trong vnstock version này.")
            return pd.DataFrame()

        df_out = pd.DataFrame()

        # 1) Thử lấy nhiều mã cùng lúc (nếu method hỗ trợ tham số symbols/tickers).
        try:
            try:
                df_out = probe.price_board(symbols=tickers_list)
            except TypeError:
                df_out = probe.price_board(tickers_list)
        except Exception:
            df_out = pd.DataFrame()

        # 2) Fallback: lấy lần lượt từng mã nếu không có multi.
        if df_out is None or (hasattr(df_out, "empty") and df_out.empty):
            frames: list[pd.DataFrame] = []
            for t in tickers_list:
                try:
                    q = Quote(source=PRIMARY_QUOTE_SOURCE, symbol=t)
                    sub = q.price_board()
                    if sub is None or (hasattr(sub, "empty") and sub.empty):
                        continue
                    if not isinstance(sub, pd.DataFrame):
                        sub = pd.DataFrame(sub)
                    sub = sub.copy()
                    if "ticker" not in sub.columns:
                        sub["ticker"] = t
                    frames.append(sub)
                except Exception as e:
                    logger.warning("price_board lỗi %s: %s", t, e)

            if frames:
                df_out = pd.concat(frames, ignore_index=True)

        if df_out is None or (hasattr(df_out, "empty") and df_out.empty):
            return pd.DataFrame()

        if not isinstance(df_out, pd.DataFrame):
            df_out = pd.DataFrame(df_out)

        # Chỉ giữ các cột cần nếu có.
        col_map = {_norm_col(c): c for c in df_out.columns}
        required_norms = [_norm_col(k) for k in required_norm_map.keys()]
        keep_cols = [col_map[rn] for rn in required_norms if rn in col_map]
        out = df_out.loc[:, keep_cols].copy() if keep_cols else df_out.copy()

        # Đảm bảo cột ticker luôn có.
        if "ticker" not in [_norm_col(c) for c in out.columns] and "ticker" not in out.columns:
            out["ticker"] = tickers_list[0]

        return out

    except Exception as e:
        logger.warning("fetch_price_board lỗi: %s", e)
        return pd.DataFrame()


# --- Run price_board snapshot ---
logger.info("⏸  Chờ %ss trước khi chuyển sang price_board...", DELAY_BETWEEN_CATEGORIES_SEC)
time.sleep(DELAY_BETWEEN_CATEGORIES_SEC)

_ensure_ingestion_workdir()
run_dt = date.today()
run_date = run_dt.isoformat()

logger.info("=" * 70)
logger.info("PRICE BOARD SNAPSHOT | %s mã", len(TICKERS))
logger.info("=" * 70)

try:
    board_df = fetch_price_board(TICKERS)
    if board_df.empty:
        logger.warning("price_board_snapshot rỗng — bỏ qua lưu.")
    else:
        save_raw(board_df, "price_board", run_date, "price_board_snapshot")
        logger.info("✓ Đã lưu price_board snapshot\n")
except Exception:
    logger.exception("✗ Lỗi khi thu thập/lưu price_board")

logger.info("=" * 70)
logger.info("✓ INGESTION HOÀN TẤT")
logger.info("=" * 70)

2026-04-04 16:05:43 [INFO] ⏸  Chờ 60s trước khi chuyển sang price_board...
2026-04-04 16:06:43 [INFO] ======================================================================
2026-04-04 16:06:43 [INFO] PRICE BOARD SNAPSHOT | 20 mã
2026-04-04 16:06:43 [INFO] ======================================================================
2026-04-04 16:06:43 [INFO] Ngoài giờ giao dịch (now=16:06:43) — bỏ qua price_board.
2026-04-04 16:06:43 [WARNING] price_board_snapshot rỗng — bỏ qua lưu.
2026-04-04 16:06:43 [INFO] ======================================================================
2026-04-04 16:06:43 [INFO] ✓ INGESTION HOÀN TẤT
2026-04-04 16:06:43 [INFO] ======================================================================


In [12]:
# CELL 9 — Kiểm tra tổng kết dữ liệu theo từng category.

import logging
from datetime import date
from pathlib import Path

import pandas as pd

logger = globals().get("logger", logging.getLogger(__name__))


def _datalake_raw_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "ingest_price_data.ipynb").is_file():
        base = cwd
    elif (cwd / "ingestion" / "ingest_price_data.ipynb").is_file():
        base = cwd / "ingestion"
    else:
        base = cwd
    return (base / ".." / "data-lake" / "raw").resolve()


def _get_null_columns(df: pd.DataFrame, threshold: float = 50.0) -> list[str]:
    """Trả về danh sách cột có tỷ lệ null > threshold%."""
    null_pct = (df.isnull().sum() / len(df) * 100)
    return [col for col, pct in null_pct.items() if pct > threshold]


raw_root = _datalake_raw_root()
run_date = date.today().isoformat()

categories = {
    "price":           f"../data-lake/raw/price/date={run_date}",
    "index":           f"../data-lake/raw/index/date={run_date}",
    "listing":         f"../data-lake/raw/listing/master",
    "company":         f"../data-lake/raw/company/date={run_date}",
    "financial_ratio": f"../data-lake/raw/financial_ratio/date={run_date}",
    "price_board":     f"../data-lake/raw/price_board/date={run_date}",
}


def _read_parquet_rows(p: Path) -> int:
    try:
        df = pd.read_parquet(p)
        return int(len(df))
    except Exception as e:
        logger.warning("QC: đọc lỗi %s: %s", p, e)
        return 0


def _get_null_cols_for_file(p: Path) -> str:
    """Đọc file parquet và trả về danh sách cột nullvalue > 50%."""
    try:
        df = pd.read_parquet(p)
        null_cols = _get_null_columns(df, threshold=50.0)
        if null_cols:
            return ", ".join(null_cols)
        else:
            return ""
    except Exception as e:
        logger.warning("QC: đọc null_cols lỗi %s: %s", p, e)
        return ""


summary_rows: list[dict] = []
missing_or_empty: list[str] = []

for cat, path_hint in categories.items():
    total_rows = 0
    file_count = 0
    null_cols_list: list[str] = []

    if cat == "listing":
        p = raw_root / "listing" / "master" / "listing.parquet"
        if p.is_file():
            file_count = 1
            total_rows = _read_parquet_rows(p)
            null_cols_list.append(_get_null_cols_for_file(p))
    else:
        folder = raw_root / cat / f"date={run_date}"
        if folder.is_dir():
            parquets = sorted(folder.glob("*.parquet"))
            file_count = len(parquets)
            total_rows = sum(_read_parquet_rows(x) for x in parquets)
            # Lấy null_cols từ file đầu tiên để represent category
            if parquets:
                null_cols_list.append(_get_null_cols_for_file(parquets[0]))

    if file_count == 0:
        status = "MISSING"
    elif total_rows == 0:
        status = "EMPTY"
    else:
        status = "OK"

    if status != "OK":
        missing_or_empty.append(cat)

    null_cols_str = "; ".join([s for s in null_cols_list if s])

    summary_rows.append(
        {
            "Category": cat,
            "Files": file_count,
            "Total rows": total_rows,
            "Status": status,
            "Null cols (>50%)": null_cols_str if null_cols_str else "-",
        }
    )


print("Category           | Files | Total rows | Status | Null cols (>50%)")
print("------------------|-------|------------|--------|------------------")
for r in summary_rows:
    null_str = r["Null cols (>50%)"]
    print(
        f"{r['Category']:<18} | {r['Files']:<5} | {r['Total rows']:<10,} | {r['Status']:<6} | {null_str}"
    )

if missing_or_empty:
    print("\n[QC WARNING] Các category bị thiếu hoặc rỗng:")
    for cat in missing_or_empty:
        print(f"- {cat}")
else:
    print("\n[QC] Tất cả category có dữ liệu.")

Category           | Files | Total rows | Status | Null cols (>50%)
------------------|-------|------------|--------|------------------
price              | 20    | 14,961     | OK     | value
index              | 5     | 3,740      | OK     | value
listing            | 1     | 4,632      | OK     | -
company            | 1     | 20         | OK     | -
financial_ratio    | 20    | 631        | OK     | -
price_board        | 0     | 0          | MISSING | -

[QC WARNING] Các category bị thiếu hoặc rỗng:
- price_board
